# Chihuahua vs Muffin — CNN Classifier

Modelo CNN capaz de distinguir entre **Chihuahua**, **Muffin** o **Ninguno de los dos**.

**Estrategia:** Entrenamos un modelo CNN binario (chihuahua vs muffin) con data augmentation, dropout y batch normalization. Para detectar imágenes que no son ni chihuahua ni muffin, usamos un **umbral de confianza**: si el modelo no está suficientemente seguro de ninguna clase, se clasifica como "Ninguno".

In [9]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

print("TensorFlow version:", tf.__version__)
print("GPU disponible:", tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.21.0
GPU disponible: []


## 1. Configuración del dataset local
El notebook está dentro de `entrega` y el dataset está en `entrega/dataset_chihuahua_muffin_clean`. Se usa esa ruta directamente.

In [10]:
# Configuración
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
EPOCHS = 30
VAL_SPLIT = 0.10
CONFIDENCE_THRESHOLD = 0.70  # Umbral para clasificar como "Ninguno"

try:
    NOTEBOOK_DIR = Path(__vsc_ipynb_file__).parent
except NameError:
    NOTEBOOK_DIR = Path(__file__).parent

DATA_DIR = NOTEBOOK_DIR / "dataset_chihuahua_muffin_clean"

if not DATA_DIR.exists():
    raise FileNotFoundError(f"No se encontró el dataset en: {DATA_DIR.resolve()}")

print("Dataset local:", DATA_DIR.resolve())


Dataset local: C:\Users\usuario\Desktop\IABigData\repos\chihuahua_vs_muffin\entrega\dataset_chihuahua_muffin_clean


## 2. Validar estructura del dataset
El dataset debe contener `train/chihuahua`, `train/muffin`, `test/chihuahua` y `test/muffin`. La partición de validación se obtiene desde `train` con un 10%.

In [11]:
# Validar estructura train/test del dataset local
classes = ["chihuahua", "muffin"]

train_dirs = [DATA_DIR / "train" / cls for cls in classes]
test_dirs = [DATA_DIR / "test" / cls for cls in classes]
missing_dirs = [path for path in train_dirs + test_dirs if not path.exists()]

if missing_dirs:
    raise FileNotFoundError(
        "Faltan carpetas dentro del dataset:\n" +
        "\n".join(str(path.resolve()) for path in missing_dirs)
    )

print("Estructura del dataset validada correctamente.")
print(f"Se usará validation_split={VAL_SPLIT:.0%} sobre la carpeta train.")

for split in ["train", "test"]:
    for cls in classes:
        split_dir = DATA_DIR / split / cls
        count = len(list(split_dir.glob("*")))
        print(f"  {split}/{cls}: {count} imágenes")

Estructura del dataset validada correctamente.
Se usará validation_split=10% sobre la carpeta train.
  train/chihuahua: 2131 imágenes
  train/muffin: 2163 imágenes
  test/chihuahua: 534 imágenes
  test/muffin: 542 imágenes


## 3. Cargar dataset y visualización

In [12]:
# Cargar datasets
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / 'train',
    validation_split=VAL_SPLIT,
    subset='training',
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / 'train',
    validation_split=VAL_SPLIT,
    subset='validation',
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / 'test', image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False
)
class_names = train_ds.class_names
print("Clases:", class_names)

# Optimizar rendimiento con prefetch
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

Found 4294 files belonging to 2 classes.
Using 3865 files for training.
Found 4294 files belonging to 2 classes.
Using 429 files for validation.
Found 1076 files belonging to 2 classes.
Clases: ['chihuahua', 'muffin']


## 4. Arquitectura CNN

**Mejoras sobre el baseline:**
- **Data Augmentation** integrado en el modelo (flip, rotación, zoom, contraste)
- **Batch Normalization** después de cada convolución
- **Dropout** para regularización
- 4 bloques convolucionales con filtros crecientes (32 → 64 → 128 → 256)
- Capas densas con dropout antes de la salida sigmoid

In [13]:
# Data Augmentation layer
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomContrast(0.1),
], name="data_augmentation")

# Modelo CNN con Batch Normalization y Dropout
model = tf.keras.Sequential([
    # Normalización de píxeles [0-255] → [0-1]
    tf.keras.layers.Rescaling(1./255, input_shape=(*IMG_SIZE, 3)),

    # Data Augmentation (solo activo durante entrenamiento)
    data_augmentation,

    # Bloque 1
    tf.keras.layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),

    # Bloque 2
    tf.keras.layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),

    # Bloque 3
    tf.keras.layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),

    # Bloque 4
    tf.keras.layers.Conv2D(256, (3, 3), padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),

    # Clasificador
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation='sigmoid')  # Salida binaria: 0=chihuahua, 1=muffin
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_1 (Rescaling)         │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 16, 16, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 16, 16, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 8, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 16384)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 256)            │     4,194,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,618,945 (17.62 MB)

 Trainable params: 4,617,473 (17.61 MB)

 Non-trainable params: 1,472 (5.75 KB)

## 5. Entrenamiento con callbacks

- **EarlyStopping**: para si no mejora en 5 épocas, restaura mejores pesos
- **ReduceLROnPlateau**: reduce learning rate si la validación se estanca

In [15]:
import time
from datetime import datetime

class TimeStopping(tf.keras.callbacks.Callback):
    """Detiene el entrenamiento si se supera el tiempo máximo."""
    def __init__(self, max_seconds=600):
        super().__init__()
        self.max_seconds = max_seconds
        self._start_time = None

    def on_train_begin(self, logs=None):
        self._start_time = time.time()

    def on_epoch_end(self, epoch, logs=None):
        elapsed = time.time() - self._start_time
        if elapsed >= self.max_seconds:
            print(f"\nTiempo límite alcanzado ({elapsed/60:.1f} min). Deteniendo entrenamiento.")
            self.model.stop_training = True

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1
    ),
    TimeStopping(max_seconds=600),  # Máximo 10 minutos
]

# Añadir TensorBoard si está disponible
try:
    import tensorboard  # noqa: F401
    tb_log_dir = NOTEBOOK_DIR / "tensorboard_logs" / f"fit_{datetime.now().strftime('%Y%m%d-%H%M%S')}"
    tb_log_dir.mkdir(parents=True, exist_ok=True)
    callbacks.append(tf.keras.callbacks.TensorBoard(log_dir=str(tb_log_dir), histogram_freq=1))
    print(f"TensorBoard activo → {tb_log_dir}")
except ImportError:
    print("TensorBoard no disponible. Instala con: pip install tensorboard")

# Entrenar
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

print(f"\nMejor val_accuracy: {max(history.history['val_accuracy']):.4f}")
print(f"Mejor val_loss: {min(history.history['val_loss']):.4f}")


TensorBoard activo → c:\Users\usuario\Desktop\IABigData\repos\chihuahua_vs_muffin\entrega\tensorboard_logs\fit_20260316-181141
Epoch 1/30


121/121 ━━━━━━━━━━━━━━━━━━━━ 56s 461ms/step - accuracy: 0.7829 - loss: 0.5223 - val_accuracy: 0.5035 - val_loss: 1.6817 - learning_rate: 0.0010
Epoch 2/30
121/121 ━━━━━━━━━━━━━━━━━━━━ 54s 443ms/step - accuracy: 0.8380 - loss: 0.4124 - val_accuracy: 0.5035 - val_loss: 3.1950 - learning_rate: 0.0010
Epoch 3/30
121/121 ━━━━━━━━━━━━━━━━━━━━ 53s 441ms/step - accuracy: 0.8533 - loss: 0.3682 - val_accuracy: 0.5594 - val_loss: 1.1971 - learning_rate: 0.0010
Epoch 4/30
121/121 ━━━━━━━━━━━━━━━━━━━━ 54s 442ms/step - accuracy: 0.8737 - loss: 0.3138 - val_accuracy: 0.8648 - val_loss: 0.3134 - learning_rate: 0.0010
Epoch 5/30
121/121 ━━━━━━━━━━━━━━━━━━━━ 54s 448ms/step - accuracy: 0.8880 - loss: 0.2785 - val_accuracy: 0.8788 - val_loss: 0.3001 - learning_rate: 0.0010
Epoch 6/30
121/121 ━━━━━━━━━━━━━━━━━━━━ 54s 443ms/step - accuracy: 0.8934 - loss: 0.2642 - val_accuracy: 0.9044 - val_loss: 0.2580 - learning_rate: 0.0010
Epoch 7/30
121/121 ━━━━━━━━━━━━━━━━━━━━ 53s 441ms/step - accuracy: 0.9030 - loss:

## 9. Guardar el modelo entrenado

In [16]:
import json

# Guardar modelo en formato Keras
model.save(NOTEBOOK_DIR / 'modelo_chihuahua_vs_muffin.keras')
print("Modelo guardado como 'modelo_chihuahua_vs_muffin.keras'")

# Guardar historial para el notebook de visualización
history_path = NOTEBOOK_DIR / 'training_history.json'
with open(history_path, 'w') as f:
    json.dump({k: [float(v) for v in vals] for k, vals in history.history.items()}, f)
print(f"Historial guardado en '{history_path.name}'")


Modelo guardado como 'modelo_chihuahua_vs_muffin.keras'
Historial guardado en 'training_history.json'
